# Phase 3 - Generate VDD foggy with Pix2Pix

Apply both trained Pix2Pix generators (medium and dense fog) to all VDD images,
at two different scales (256 and 768), producing 4 foggy datasets:

    VDD_foggy_medium_256  (apply 256, save 768)
    VDD_foggy_medium_768  (apply 768, save 768)
    VDD_foggy_dense_256   (apply 256, save 768)
    VDD_foggy_dense_768   (apply 768, save 768)

All foggy images are saved at 768x768 (the U-Net's eval resolution) so the
next phase can read them directly without further resizing.

Total time on T4: ~5-10 minutes (inference is fast).

**Before running**: Runtime -> Change runtime type -> T4 GPU.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone the project from GitHub

In [ ]:
import os

!rm -rf /content/fog-uav-robustness
!git clone https://github.com/FabrizioCozzolino/fog-uav-robustness.git /content/fog-uav-robustness
%cd /content/fog-uav-robustness
!ls

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/fog-uav-robustness'
!ls -la "{DRIVE}/outputs"

Expected: you should see `pix2pix_medium_baseline/` and `pix2pix_dense_baseline/`
from Phase 2, plus `unet_resnet34_clean_v2_weighted/` from Phase 1.

## 4. Copy VDD from Drive

Same as Phase 1: we read VDD images at full resolution from local SSD
(faster than network mount). 5-8 minutes for ~2 GB.

In [ ]:
import os, time, shutil

DRIVE_VDD = '/content/drive/MyDrive/fog-uav-robustness/data/VDD'
LOCAL_VDD = '/content/data/VDD'

if os.path.isdir(os.path.join(DRIVE_VDD, 'VDD', 'train')):
    src = os.path.join(DRIVE_VDD, 'VDD')
elif os.path.isdir(os.path.join(DRIVE_VDD, 'train')):
    src = DRIVE_VDD
else:
    raise RuntimeError(f'Could not find train/ inside {DRIVE_VDD}.')

if os.path.isdir(LOCAL_VDD):
    shutil.rmtree(LOCAL_VDD)
os.makedirs('/content/data', exist_ok=True)

print(f'Copying from {src} (~5-8 minutes)')
t0 = time.time()
!cp -r "{src}" "{LOCAL_VDD}"
print(f'Copy took {time.time() - t0:.0f} s')

for split in ['train', 'val', 'test']:
    n = len(os.listdir(f'{LOCAL_VDD}/{split}/src'))
    print(f'{split}: {n} images')

## 5. Restore the two Pix2Pix generators from Drive

We need both `G_best.pth` files to be in the local Colab filesystem so the
inference script can load them quickly.

In [ ]:
import os, shutil

for run_name in ['pix2pix_medium_baseline', 'pix2pix_dense_baseline']:
    SRC = f'/content/drive/MyDrive/fog-uav-robustness/outputs/{run_name}'
    DST = f'/content/fog-uav-robustness/outputs/runs/{run_name}'
    if not os.path.isdir(SRC):
        raise RuntimeError(f'{SRC} not on Drive! Did Phase 2 finish and back up?')
    if os.path.isdir(DST):
        shutil.rmtree(DST)
    os.makedirs(os.path.dirname(DST), exist_ok=True)
    shutil.copytree(SRC, DST)
    print(f'[ok] restored {run_name}')
    !ls -la "{DST}" | head -8

## 6. Install Python dependencies

In [ ]:
!pip install -q -r requirements-colab.txt

## 7. Generate the four foggy datasets

Each call produces a separate folder with 280+80+40 = 400 foggy images and
their (unchanged) masks. Output is saved at 768x768 in JPG (~95 quality)
for compactness.

In [ ]:
# 7a. medium fog, applied at 256x256 (training resolution of Pix2Pix)
!python src/inference/generate_foggy_vdd.py \
    --generator outputs/runs/pix2pix_medium_baseline/G_best.pth \
    --vdd-root /content/data/VDD \
    --output-root /content/data/VDD_foggy_medium_256 \
    --apply-size 256 \
    --save-size 768 \
    --batch-size 4

In [ ]:
# 7b. medium fog, applied at 768x768 (matches U-Net eval resolution)
!python src/inference/generate_foggy_vdd.py \
    --generator outputs/runs/pix2pix_medium_baseline/G_best.pth \
    --vdd-root /content/data/VDD \
    --output-root /content/data/VDD_foggy_medium_768 \
    --apply-size 768 \
    --save-size 768 \
    --batch-size 4

In [ ]:
# 7c. dense fog, applied at 256x256
!python src/inference/generate_foggy_vdd.py \
    --generator outputs/runs/pix2pix_dense_baseline/G_best.pth \
    --vdd-root /content/data/VDD \
    --output-root /content/data/VDD_foggy_dense_256 \
    --apply-size 256 \
    --save-size 768 \
    --batch-size 4

In [ ]:
# 7d. dense fog, applied at 768x768
!python src/inference/generate_foggy_vdd.py \
    --generator outputs/runs/pix2pix_dense_baseline/G_best.pth \
    --vdd-root /content/data/VDD \
    --output-root /content/data/VDD_foggy_dense_768 \
    --apply-size 768 \
    --save-size 768 \
    --batch-size 4

## 8. Sanity check: visualize a few foggy samples

Pick one VDD test image and show its 4 foggy variants (clean | med-256 | med-768
| dense-256 | dense-768) in a single figure, so we can compare. This is what
the U-Net will see in Phase 4.

In [ ]:
import os, cv2, matplotlib.pyplot as plt

TEST_NAMES = ['DJI_0009', 'DJI_0357', 'DJI_0469', 'DJI_0917']  # a few test images
VARIANTS = [
    ('clean (original)',        '/content/data/VDD/test/src',                    '.JPG'),
    ('foggy medium @256',       '/content/data/VDD_foggy_medium_256/test/src',   '.JPG'),
    ('foggy medium @768',       '/content/data/VDD_foggy_medium_768/test/src',   '.JPG'),
    ('foggy dense  @256',       '/content/data/VDD_foggy_dense_256/test/src',    '.JPG'),
    ('foggy dense  @768',       '/content/data/VDD_foggy_dense_768/test/src',    '.JPG'),
]

fig, axes = plt.subplots(len(TEST_NAMES), len(VARIANTS), figsize=(4 * len(VARIANTS), 3 * len(TEST_NAMES)))
for i, name in enumerate(TEST_NAMES):
    for j, (title, folder, ext) in enumerate(VARIANTS):
        path = os.path.join(folder, name + ext)
        if not os.path.isfile(path):
            axes[i, j].set_title(f'{title}\n[missing]', fontsize=9)
            axes[i, j].axis('off')
            continue
        img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
        axes[i, j].imshow(img)
        if i == 0:
            axes[i, j].set_title(title, fontsize=10)
        if j == 0:
            axes[i, j].set_ylabel(name, fontsize=9)
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

os.makedirs('outputs/figures', exist_ok=True)
out_path = 'outputs/figures/vdd_foggy_comparison.png'
plt.tight_layout()
plt.savefig(out_path, dpi=80, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 9. Backup the foggy datasets to Drive

Copy the four `VDD_foggy_*` folders to Drive so we don't lose them when the
Colab session ends. ~200 MB each, total ~800 MB.

*Tip*: if Drive is short on space, keep only the two 768-versions (the ones
we'll likely use in Phase 4) and skip the 256-versions, which take more space
than they're worth (already upscaled in storage).

In [ ]:
import os, shutil

DRIVE_DATA = '/content/drive/MyDrive/fog-uav-robustness/data'
datasets = [
    'VDD_foggy_medium_256',
    'VDD_foggy_medium_768',
    'VDD_foggy_dense_256',
    'VDD_foggy_dense_768',
]
for ds in datasets:
    SRC = f'/content/data/{ds}'
    DST = f'{DRIVE_DATA}/{ds}'
    if not os.path.isdir(SRC):
        print(f'[skip] {SRC} not found locally')
        continue
    print(f'Copying {ds} -> Drive ...')
    if os.path.isdir(DST):
        shutil.rmtree(DST)
    shutil.copytree(SRC, DST)
    n_train = len(os.listdir(f'{DST}/train/src'))
    n_val   = len(os.listdir(f'{DST}/val/src'))
    n_test  = len(os.listdir(f'{DST}/test/src'))
    print(f'  [ok] {ds}: train={n_train}, val={n_val}, test={n_test}')
print('\nAll backed up to Drive.')

## What's next (Phase 4)

We will run our trained U-Net (Phase 1's v2 model) on each of the four foggy
test sets and compare:

| dataset                    | mIoU (test) | drop vs clean |
|----------------------------|-------------|---------------|
| VDD clean                  | 0.7168      | (baseline)    |
| VDD_foggy_medium_256       | ?           | ?             |
| VDD_foggy_medium_768       | ?           | ?             |
| VDD_foggy_dense_256        | ?           | ?             |
| VDD_foggy_dense_768        | ?           | ?             |

This is the central result of the project: the **performance drop curve** as
fog severity increases. We expect dense > medium drop, and we'll learn whether
the apply-size choice matters.